In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import RandomizedSearchCV
import pickle as pkl

In [2]:
df = pd.read_csv("../data/processed data/final_df.csv")

In [3]:
df.isnull().values.any()

np.False_

In [4]:
list[df.columns]

list[Index(['Unnamed: 0', 'Brand', 'Model', 'Series', 'Weight', 'Operating System',
       'Operating System Type', 'Display Size', 'Resolution_X', 'Resolution_Y',
       'Pixel Density', 'cpu_brand', 'cpu_model', 'Display Touchscreen',
       'Cache', 'Graphic Processor', 'Capacity', 'SSD Capacity', 'SSD Type',
       'Graphics Memory', 'HDD Capacity', 'HDD Type', 'Battery Life',
       'market_status', 'Price (Rs)'],
      dtype='str')]

In [5]:
#log transform the target
df['Price (Rs)'] = np.log1p(df['Price (Rs)'])

In [6]:
#separate features and target
X=df.drop('Price (Rs)' , axis=1)
y=df['Price (Rs)']

In [7]:
#applying train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [8]:
#model 1 = Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [9]:
y_pred_lr = lr.predict(X_test)

In [10]:
#metrics with log values
mae_lr_log = mean_absolute_error(y_test, y_pred_lr)
rmse_lr_log = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr_log = r2_score(y_test, y_pred_lr)

In [11]:
#convert back the target
y_pred_lr_actual = np.expm1(y_pred_lr)
y_test_actual = np.expm1(y_test)

In [12]:
#metrics with actual values
mae_lr_actual = mean_absolute_error(y_test_actual, y_pred_lr_actual)
rmse_lr_actual = np.sqrt(mean_squared_error(y_test_actual, y_pred_lr_actual))
r2_lr_actual = r2_score(y_test_actual, y_pred_lr_actual)

In [13]:
print("Linear Regression Results:")
print("Log Metrics")
print("MAE:", mae_lr_log)
print("RMSE:", rmse_lr_log)
print("R2 Score:", r2_lr_log)

print("Actual Metrics")
print("MAE:", mae_lr_actual)
print("RMSE:", rmse_lr_actual)
print("R2 Score:", r2_lr_actual)

Linear Regression Results:
Log Metrics
MAE: 0.2245499390730253
RMSE: 0.30308872758862737
R2 Score: 0.7566910141268456
Actual Metrics
MAE: 22302.19592704344
RMSE: 94806.020515453
R2 Score: -1.5714417614902274


In [14]:
#Model 2 = Random Forest Regressor
rf = RandomForestRegressor()
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [15]:
y_pred_rf = rf.predict(X_test)

In [16]:
#metrics with log values
mae_rf_log = mean_absolute_error(y_test, y_pred_rf)
rmse_rf_log = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf_log = r2_score(y_test, y_pred_rf)

In [17]:
#convert back the target
y_pred_rf_actual = np.expm1(y_pred_rf)

In [18]:
#metrics with actual values
mae_rf_actual = mean_absolute_error(y_test_actual, y_pred_rf_actual)
rmse_rf_actual = np.sqrt(mean_squared_error(y_test_actual, y_pred_rf_actual))
r2_rf_actual = r2_score(y_test_actual, y_pred_rf_actual)

In [19]:
print("Random Forest Results:")
print("Log Metrics")
print("MAE:", mae_rf_log)
print("RMSE:", rmse_rf_log)
print("R2 Score:", r2_rf_log)

print("Actual Metrics")
print("MAE:", mae_rf_actual)
print("RMSE:", rmse_rf_actual)
print("R2 Score:", r2_rf_actual)

Random Forest Results:
Log Metrics
MAE: 0.14132937242868862
RMSE: 0.1982941477682868
R2 Score: 0.8958550121056432
Actual Metrics
MAE: 12431.483596340644
RMSE: 22638.080416420125
R2 Score: 0.8533830926436516


In [20]:
pkl.dump(rf, open('random_forest_model.pkl', 'wb'))